In [43]:
import pandas as pd
import polars as pl
import numpy as np
import xgboost as xgb
from src import exploration_cleaning_methods as ecm
from src import prediction_methods as pm

**Men**

In [44]:
m_train_df: pl.DataFrame = pl.read_parquet("../data/proccessed/m_train_data.parquet")
m_train_df.head(10)

Season,ATeamID,BTeamID,DayNum,Target,TeamScore_Diff,OpponentScore_Diff,WinRatio_Diff,Location_Diff,TeamPOS_Diff,OpponentPOS_Diff,OffEfficiency_Diff,DefEfficiency_Diff,TeamEFG_Diff,OpponentEFG_Diff,TurnoverRate_Diff,Seed_Diff
i16,i16,i16,i16,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
2015,1214,1264,134,1,-2.723485,0.5,-0.108902,-0.088068,0.708333,0.207718,-5.145762,1.099798,-0.040666,-0.014806,-0.007078,0
2015,1279,1140,134,1,-9.21875,-5.4375,-0.09375,0.0,-3.496094,-3.71875,-7.572157,-2.260103,-0.047807,-0.024097,0.002589,0
2015,1173,1129,135,1,-1.658847,-0.026393,0.01564,0.183773,1.108333,0.864761,-4.495298,-1.625657,-0.008537,-0.001035,0.017282,0
2015,1352,1316,135,1,-4.935484,-0.788856,-0.069404,0.225806,-0.187488,-0.478739,-6.909791,-0.362959,-0.02873,0.003899,0.022644,0
2015,1112,1411,136,1,8.264706,-8.823529,0.264706,0.5,1.188971,1.156618,10.248704,-15.081474,0.037546,-0.04521,-0.031617,-13
2015,1116,1459,136,1,12.154412,10.084559,-0.047794,0.299632,9.121645,8.196186,2.760751,3.306361,-0.016964,0.01076,-0.006101,-7
2015,1139,1400,136,1,1.623106,0.793561,0.081439,-0.055871,1.248911,1.918821,0.588484,-1.196768,-0.000487,0.047622,-0.024455,-5
2015,1153,1345,136,1,-7.625,-9.233902,0.051136,0.006629,-5.668324,-5.262287,-2.104367,-6.399262,-0.000502,-0.007881,0.014252,-1
2015,1207,1186,136,1,-8.645161,-9.612903,-0.064516,0.387097,-3.245968,-3.291935,-7.236319,-9.348895,-0.044199,-0.053542,0.032444,-9


In [45]:
m_regular_season_detailed_lf: pl.LazyFrame = pl.scan_csv("../data/raw/MRegularSeasonDetailedResults.csv")
m_regular_season_detailed_lf.head(10).collect()

Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
i64,i64,i64,i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2003,10,1104,68,1328,62,"""N""",0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20
2003,10,1272,70,1393,63,"""N""",0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16
2003,11,1266,73,1437,61,"""N""",0,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23
2003,11,1296,56,1457,50,"""N""",0,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23
2003,11,1400,77,1208,71,"""N""",0,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14
2003,11,1458,81,1186,55,"""H""",0,26,57,6,12,23,27,12,24,12,9,9,3,18,20,46,3,11,12,17,6,22,8,19,4,3,25
2003,12,1161,80,1236,62,"""H""",0,23,55,2,8,32,39,13,18,14,17,11,1,25,19,41,4,15,20,28,9,21,11,30,10,4,28
2003,12,1186,75,1457,61,"""N""",0,28,62,4,14,15,21,13,35,19,19,7,2,21,20,59,4,17,17,23,8,25,10,15,14,8,18
2003,12,1194,71,1156,66,"""N""",0,28,58,5,11,10,18,9,22,9,17,9,2,23,24,52,6,18,12,27,13,26,13,25,8,2,18


In [46]:
m_prof_df: pl.DataFrame = ecm.create_team_season_profile(
    m_regular_season_detailed_lf.filter(pl.col("Season") == 2026)
).collect()

In [47]:
m_grid_df: pl.DataFrame = pm.create_carthesian_matchup_grid(
    m_regular_season_detailed_lf
    .filter(pl.col("Season") == 2026)
    .collect()
)

In [48]:
del m_regular_season_detailed_lf

In [49]:
m_infer_df: pl.DataFrame = (
    m_grid_df
    .join(
        m_prof_df, 
        left_on=["Season", "ATeamID"], 
        right_on=["Season", "TeamID"], 
        how="left"
    )
    .join(
        m_prof_df, 
        left_on=["Season", "BTeamID"], 
        right_on=["Season", "TeamID"], 
        how="left", 
        suffix="_B"
))

In [50]:
base_cols: list[str] = [
    "TeamScore", "OpponentScore", "WinRatio", "Location", "TeamPOS", 
    "OpponentPOS", "OffEfficiency", "DefEfficiency", "TeamEFG", 
    "OpponentEFG", "TurnoverRate"
]

for col in base_cols:
    m_infer_df = m_infer_df.with_columns((pl.col(col) - pl.col(f"{col}_B")).alias(f"{col}_Diff"))

In [51]:
del m_grid_df, m_prof_df

In [52]:
m_model: xgb.Booster = pm.generate_model(m_train_df)
feat_cols: list[str] = [f"{col}_Diff" for col in base_cols]
m_sub: pl.DataFrame = pm.predict_and_create_submission_data(m_model, feat_cols, m_infer_df)
m_sub.head(10)

ID,Pred
str,f32
"""2026_1462_1477""",0.943773
"""2026_1462_1465""",0.408792
"""2026_1462_1471""",0.526872
"""2026_1462_1480""",0.866596
"""2026_1462_1474""",0.537494
"""2026_1462_1468""",0.870436
"""2026_1462_1469""",0.782557
"""2026_1462_1475""",0.789253
"""2026_1462_1481""",0.825404


**Women**

In [53]:
w_train_df: pl.DataFrame = pl.read_parquet("../data/proccessed/w_train_data.parquet")
w_train_df.head(10)

Season,ATeamID,BTeamID,DayNum,Target,TeamScore_Diff,OpponentScore_Diff,WinRatio_Diff,Location_Diff,TeamPOS_Diff,OpponentPOS_Diff,OffEfficiency_Diff,DefEfficiency_Diff,TeamEFG_Diff,OpponentEFG_Diff,TurnoverRate_Diff,Seed_Diff
i16,i16,i16,i16,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
2015,3116,3321,137,1,-12.022581,-7.503226,-0.175269,0.133333,-8.202715,-8.009247,-5.922198,-0.89886,-0.065267,-0.022153,0.002366,3
2015,3124,3322,137,1,20.336468,-3.606061,0.39185,0.492163,7.632576,5.852038,17.930723,-12.731083,0.06781,-0.053646,-0.008088,-13
2015,3143,3455,137,1,5.137311,11.217803,-0.160038,-0.119318,8.246875,8.141738,-4.819232,6.824334,0.002632,0.000894,0.012675,-9
2015,3173,3235,137,1,6.274194,0.672043,0.206452,-0.232258,6.075887,5.723441,0.895106,-6.573663,0.010844,-0.037637,0.030525,-3
2015,3177,3278,137,1,10.475379,-1.839962,0.069129,0.058712,3.560014,4.395952,9.203691,-7.623859,0.028656,0.038958,-0.047683,1
2015,3181,3107,137,1,1.367944,3.879032,-0.072581,0.221774,3.045514,3.76376,-2.771438,1.139297,-0.018015,-0.041011,0.009153,-9
2015,3211,3203,137,1,-1.354839,2.774194,-0.129032,-0.032258,-5.674194,-4.784677,5.71733,9.253443,0.030676,0.016624,0.003185,5
2015,3234,3110,137,1,16.462702,15.394153,0.024194,0.09879,9.587399,9.258191,9.314946,9.813344,0.03694,0.048403,-0.012815,-11
2015,3246,3398,137,1,7.008333,1.329167,0.11875,0.091667,3.79724,4.269635,4.041734,-3.23085,0.001469,-0.000361,-0.013834,-13


In [54]:
w_regular_season_detailed_lf: pl.LazyFrame = pl.scan_csv("../data/raw/WRegularSeasonDetailedResults.csv")
w_regular_season_detailed_lf.head(10).collect

<bound method LazyFrame.collect of <LazyFrame at 0x254C0E9B8E0>>

In [55]:
w_prof_df: pl.DataFrame = ecm.create_team_season_profile(
    w_regular_season_detailed_lf.filter(pl.col("Season") == 2026)
).collect()

In [56]:
w_grid_df: pl.DataFrame = pm.create_carthesian_matchup_grid(
    w_regular_season_detailed_lf
    .filter(pl.col("Season") == 2026)
    .collect()
)

In [57]:
del w_regular_season_detailed_lf

In [58]:
w_infer_df: pl.DataFrame = (
    w_grid_df
    .join(
        w_prof_df, 
        left_on=["Season", "ATeamID"], 
        right_on=["Season", "TeamID"], 
        how="left"
    )
    .join(
        w_prof_df, 
        left_on=["Season", "BTeamID"], 
        right_on=["Season", "TeamID"], 
        how="left", 
        suffix="_B"
))

In [59]:
base_cols: list[str] = [
    "TeamScore", "OpponentScore", "WinRatio", "Location", "TeamPOS", 
    "OpponentPOS", "OffEfficiency", "DefEfficiency", "TeamEFG", 
    "OpponentEFG", "TurnoverRate"
]

for col in base_cols:
    w_infer_df = w_infer_df.with_columns((pl.col(col) - pl.col(f"{col}_B")).alias(f"{col}_Diff"))

In [60]:
del w_grid_df, w_prof_df

In [61]:
w_model: xgb.Booster = pm.generate_model(w_train_df)
feat_cols: list[str] = [f"{col}_Diff" for col in base_cols]
w_sub: pl.DataFrame = pm.predict_and_create_submission_data(w_model, feat_cols, w_infer_df)
w_sub.head(10)

ID,Pred
str,f32
"""2026_3177_3278""",0.030071
"""2026_3177_3430""",0.58564
"""2026_3177_3180""",0.241172
"""2026_3177_3308""",0.85308
"""2026_3177_3290""",0.870111
"""2026_3177_3299""",0.847909
"""2026_3177_3448""",0.033302
"""2026_3177_3427""",0.457441
"""2026_3177_3454""",0.408166
